# 04 — Resource scaling

Compares the direct edge-QUBO and path-QUBO as the number of services, time cycles, satellites and retained candidate paths changes.

In [ ]:
from __future__ import annotations

from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from oec_qaoa.edge_qubo import build_edge_qubo
from oec_qaoa.generators import SyntheticConfig, synthetic_relay_instance
from oec_qaoa.metrics import qubo_resource_metrics
from oec_qaoa.path_qubo import build_path_qubo
from oec_qaoa.path_reduction import candidate_sets

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS = ROOT / "results" / "scaling"
RESULTS.mkdir(parents=True, exist_ok=True)

## Controlled synthetic family

The synthetic family contains one acquisition satellite, configurable relay satellites and one virtual ground station. Services may be stored, processed on board or delivered unprocessed for ground processing. The parameters varied are `n_satellites`, `n_time_cycles`, `n_services`, `K` and a shared-resource capacity factor.

In [ ]:
BASE = {
    "n_satellites": 4,
    "n_time_cycles": 5,
    "n_services": 4,
    "capacity_factor": 1.0,
    "seed": 20260929,
}

example = synthetic_relay_instance(SyntheticConfig(**BASE))
print(example.name)
print(f"Services: {len(example.services)}")
print(f"Graph edges: {len(example.edges)}")

## Resource metrics

For each QUBO we record semantic, slack and total logical variables, the number of non-zero quadratic couplings `N_ZZ`, interaction density, graph degree statistics and coefficient dynamic range.

In [ ]:
edge_model = build_edge_qubo(example)
path_model = build_path_qubo(example, candidate_sets(example, k=2))

pd.DataFrame(
    [
        {
            "formulation": "edge",
            **qubo_resource_metrics(edge_model).to_dict(),
        },
        {
            "formulation": "path, K=2",
            **qubo_resource_metrics(path_model).to_dict(),
        },
    ]
)

## One-parameter sweeps

Each structural parameter is varied around a common baseline to isolate its contribution to the encoding size.

In [ ]:
sweeps = {
    "n_services": [2, 4, 6, 8, 10, 12],
    "n_time_cycles": [3, 4, 5, 6, 8],
    "n_satellites": [3, 4, 5, 6, 8],
}

K_VALUES = [1, 2, 4, 8]
SEEDS = [20260929, 20260930, 20261001]

records = []

for parameter, values in sweeps.items():
    for value in values:
        for seed in SEEDS:
            configuration = dict(BASE)
            configuration[parameter] = value
            configuration["seed"] = seed

            instance = synthetic_relay_instance(
                SyntheticConfig(**configuration)
            )

            edge_qubo = build_edge_qubo(instance)
            edge_metrics = qubo_resource_metrics(edge_qubo)

            records.append(
                {
                    "sweep": parameter,
                    "sweep_value": value,
                    "seed": seed,
                    "formulation": "edge",
                    "K": None,
                    **configuration,
                    **edge_metrics.to_dict(),
                }
            )

            all_paths = candidate_sets(instance)
            max_available = min(
                len(paths)
                for paths in all_paths.values()
            )

            for k in K_VALUES:
                if k > max_available:
                    continue

                paths = {
                    service: candidates[:k]
                    for service, candidates in all_paths.items()
                }
                path_qubo = build_path_qubo(instance, paths)
                path_metrics = qubo_resource_metrics(path_qubo)

                records.append(
                    {
                        "sweep": parameter,
                        "sweep_value": value,
                        "seed": seed,
                        "formulation": "path",
                        "K": k,
                        **configuration,
                        **path_metrics.to_dict(),
                    }
                )

scaling = pd.DataFrame(records)
scaling.head()

In [ ]:
output_csv = RESULTS / "resource_scaling.csv"
scaling.to_csv(output_csv, index=False)
print(output_csv)

## Aggregated trends

Means are computed across deterministic seeds for each configuration.

In [ ]:
def plot_metric(
    frame: pd.DataFrame,
    sweep: str,
    metric: str,
) -> None:
    subset = frame[frame["sweep"] == sweep].copy()
    grouped = (
        subset.groupby(
            ["sweep_value", "formulation", "K"],
            dropna=False,
        )[metric]
        .mean()
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(7.2, 4.5))

    edge = grouped[grouped["formulation"] == "edge"]
    ax.plot(
        edge["sweep_value"],
        edge[metric],
        marker="o",
        label="edge QUBO",
    )

    for k in K_VALUES:
        path = grouped[
            (grouped["formulation"] == "path")
            & (grouped["K"] == k)
        ]
        if path.empty:
            continue
        ax.plot(
            path["sweep_value"],
            path[metric],
            marker="o",
            label=f"path QUBO, K={k}",
        )

    ax.set_xlabel(sweep.replace("_", " "))
    ax.set_ylabel(metric.replace("_", " "))
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
plot_metric(scaling, "n_services", "logical_variables")
plot_metric(scaling, "n_services", "quadratic_couplings")
plot_metric(scaling, "n_services", "qubo_density")

In [ ]:
plot_metric(scaling, "n_time_cycles", "logical_variables")
plot_metric(scaling, "n_satellites", "logical_variables")

## Reduction ratios

The path-to-edge reductions are summarized by

\[
R_N=\frac{N_{\mathrm{logical}}^{\mathrm{path}}}{N_{\mathrm{logical}}^{\mathrm{edge}}},\qquad
R_{ZZ}=\frac{N_{ZZ}^{\mathrm{path}}}{N_{ZZ}^{\mathrm{edge}}}.
\]

In [ ]:
edge_rows = scaling[
    scaling["formulation"] == "edge"
][
    [
        "sweep",
        "sweep_value",
        "seed",
        "logical_variables",
        "quadratic_couplings",
    ]
].rename(
    columns={
        "logical_variables": "edge_logical_variables",
        "quadratic_couplings": "edge_quadratic_couplings",
    }
)

path_rows = scaling[
    scaling["formulation"] == "path"
].copy()

ratios = path_rows.merge(
    edge_rows,
    on=["sweep", "sweep_value", "seed"],
    how="left",
)

ratios["logical_variable_ratio"] = (
    ratios["logical_variables"]
    / ratios["edge_logical_variables"]
)
ratios["coupling_ratio"] = (
    ratios["quadratic_couplings"]
    / ratios["edge_quadratic_couplings"]
)

ratios.to_csv(
    RESULTS / "resource_reduction_ratios.csv",
    index=False,
)

ratios[
    [
        "sweep",
        "sweep_value",
        "K",
        "logical_variable_ratio",
        "coupling_ratio",
    ]
].head()

## Combined grid

A smaller multi-parameter grid checks that the one-parameter trends persist away from the baseline configuration.

In [ ]:
combined_records = []

for n_satellites, n_time_cycles, n_services, seed in product(
    [3, 4, 6],
    [4, 6],
    [2, 4, 8],
    SEEDS,
):
    config = SyntheticConfig(
        n_satellites=n_satellites,
        n_time_cycles=n_time_cycles,
        n_services=n_services,
        capacity_factor=1.0,
        seed=seed,
    )
    instance = synthetic_relay_instance(config)

    edge = build_edge_qubo(instance)
    combined_records.append(
        {
            "formulation": "edge",
            "K": None,
            "n_satellites": n_satellites,
            "n_time_cycles": n_time_cycles,
            "n_services": n_services,
            "seed": seed,
            **qubo_resource_metrics(edge).to_dict(),
        }
    )

    all_paths = candidate_sets(instance)
    max_available = min(len(paths) for paths in all_paths.values())

    for k in (2, 4):
        if k > max_available:
            continue
        paths = {
            service: candidates[:k]
            for service, candidates in all_paths.items()
        }
        model = build_path_qubo(instance, paths)

        combined_records.append(
            {
                "formulation": "path",
                "K": k,
                "n_satellites": n_satellites,
                "n_time_cycles": n_time_cycles,
                "n_services": n_services,
                "seed": seed,
                **qubo_resource_metrics(model).to_dict(),
            }
        )

combined = pd.DataFrame(combined_records)
combined.to_csv(
    RESULTS / "resource_scaling_combined.csv",
    index=False,
)

combined.head()

## Saved outputs

The sweep tables are written to `results/scaling/` for downstream analysis.